# Inferência (ModelLoader + Predictor)

Treina um modelo minúsculo, salva, recarrega pelo `ModelLoader` e prevê
com o `Predictor` — o caminho real de detecção, incluindo a leitura do
`input_contract` (temperatura/EER) e o fallback ONNX→TF.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "app").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT)

## Reprodutibilidade (semente numpy + TensorFlow)

In [ ]:
import numpy as np

try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(0)
except Exception:  # pragma: no cover
    np.random.seed(0)

## 1. Treinar + salvar um modelo de demonstração

In [ ]:
import tempfile
import numpy as np
from app.core.interfaces.base import ProcessingStatus
from app.domain.services.training_service import TrainingService

rng = np.random.default_rng(1)
n, T, F = 240, 32, 16
X = rng.standard_normal((n, T, F)).astype("float32")
y = rng.integers(0, 2, n).astype("int64")
X[y == 1] += 0.6

workdir = Path(tempfile.mkdtemp(prefix="nb_infer_"))
npz = workdir / "ds.npz"
np.savez(npz, X_train=X, y_train=y)
models_dir = workdir / "models"
res = TrainingService(models_dir=str(models_dir)).train_model(
    architecture="MultiscaleCNN", dataset_path=str(npz),
    config={"epochs": 2, "batch_size": 32, "model_name": "nb_infer"},
)
assert res.status == ProcessingStatus.SUCCESS, res.errors
print("treinado e salvo em", models_dir)

## 2. Carregar e prever

In [ ]:
from app.domain.services.detection.model_loader import ModelLoader
from app.domain.services.detection.predictor import Predictor

loader = ModelLoader(models_dir=str(models_dir))
loader.load_available_models()
mi = loader.get_model("nb_infer")
print("modelo:", mi.architecture, "| input_shape:", mi.input_shape)
print("temperatura calibrada:", mi.temperature,
      "| eer_threshold:", mi.eer_threshold)

sample = X[:1][0]  # uma amostra (T, F)
out = Predictor().predict(mi, sample)
print("resultado:", out.data)

## Notas

- `ModelLoader` lê o `input_contract` do `_config.json` (temperatura/EER/OOD)
  e o `Predictor` os aplica automaticamente.
- Se houver um `nb_infer.onnx` ao lado e `onnxruntime` instalado, a
  inferência usa ONNX (mais rápida em CPU) com fallback para o TF.